# 02 — Supervised Fine-Tuning (Models A/B/C)
**Research Objective:** Reproduce supervised fine-tuning and evaluation on the FiReCS sentiment dataset. This notebook provides dataset preparation, tokenization, evaluation metrics (macro-F1), and optional training blocks (guarded).
## Method Overview
**Why:** Compare baseline (Model A), DAPT-initialized (Model B), and alternate architecture (Model C - XLM-R). Assumptions: preprocessed splits are reproducible with the seed; `models/*_finetuned` are canonical outputs when present.

In [5]:
# Configuration & reproducibility (single action)
ROOT = r"D:\\NLP"
DEFAULT_DATASET = "ccosme/FiReCS"
ARTIFACTS_DIR = ROOT + "\\models"
RUN_FINETUNE = True  # Set to True only when you intend to train
SEED = 42

# set_seed comes from utils inline in notebooks; if not present, import here
try:
    set_seed(SEED)
except NameError:
    import random, numpy as np, torch
    def set_seed(seed: int = SEED, deterministic: bool = False):
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
        if deterministic:
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
    set_seed(SEED)
print('Seed set to', SEED)

Seed set to 42


In [6]:
# Minimal utility: dataset preparation (single action, adapted from 04_finetune_models.py)
from datasets import load_dataset, concatenate_datasets

def prepare_firescs_splits(dataset_name=DEFAULT_DATASET, seed=SEED):
    ds = load_dataset(dataset_name)
    label_map = {"negative": 0, "positive": 1, "neutral": 2}
    def transform(example):
        if isinstance(example.get("label"), (int, float)):
            label = int(example["label"])
        else:
            label = label_map.get(example["label"].lower(), 2)
        return {"text_content": example["review"], "label_int": label}
    splits = []
    for s in ['train','test','validation']:
        if s in ds:
            splits.append(ds[s])
    full = concatenate_datasets(splits)
    full = full.map(transform, remove_columns=full.column_names)
    tmp = full.train_test_split(test_size=0.2, seed=seed)
    val_test = tmp['test'].train_test_split(test_size=0.5, seed=seed)
    return {'train': tmp['train'], 'validation': val_test['train'], 'test': val_test['test']}
print("Prepared splits (not materialized here).")

d:\NLP\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Prepared splits (not materialized here).


In [7]:
# Load dataset splits (single action)
splits = prepare_firescs_splits()
print({k: len(v) for k,v in splits.items()})

{'train': 8389, 'validation': 1049, 'test': 1049}


In [8]:
# A single, small training demo function (one logical action) — runs only if RUN_FINETUNE True
# Be resilient if the user didn't run the configuration cell: default to skipping.
try:
    RUN_FINETUNE
except NameError:
    RUN_FINETUNE = False
    print("RUN_FINETUNE not set; defaulting to False. To run training set RUN_FINETUNE=True and re-run this cell.")

if RUN_FINETUNE:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-multilingual-cased")
    def tokenize_fn(examples):
        return tokenizer(examples["text_content"], padding="max_length", truncation=True, max_length=128)
    tokenized = {k: v.map(tokenize_fn, batched=True).rename_column("label_int", "labels") for k,v in splits.items()}
    model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-multilingual-cased", num_labels=3)

    # Construct TrainingArguments in a forwards/backwards-compatible way
    args = TrainingArguments(output_dir=ARTIFACTS_DIR + "\\model_A_finetuned",
                             num_train_epochs=2,
                             per_device_train_batch_size=4,
                             report_to="none")
    # Try to enable epoch evaluation compatibly with different transformers versions
    if hasattr(args, "evaluation_strategy"):
        args.evaluation_strategy = "epoch"
    elif hasattr(args, "do_eval"):
        args.do_eval = True

    trainer = Trainer(model=model, args=args, train_dataset=tokenized["train"], eval_dataset=tokenized["validation"])
    trainer.train()
else:
    print("Fine-tuning cell skipped. To run, set RUN_FINETUNE=True and re-run this cell.")

Map: 100%|██████████| 1049/1049 [00:00<00:00, 10983.39 examples/s]
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
500,0.879900
1000,0.709200
1500,0.637500
2000,0.623500
2500,0.537500
3000,0.457600
3500,0.438300
4000,0.481500


In [9]:
# Evaluate a saved model (single action) — loads model from disk if present
from sklearn.metrics import f1_score, accuracy_score
import os
import torch

def evaluate_saved_model(path, split=None, tokenizer_name=None):
    # Ensure splits exist (if the user did not run the splits cell)
    global splits
    if 'splits' not in globals():
        print("`splits` not found; preparing dataset splits on-the-fly.")
        splits = prepare_firescs_splits()
    split = split or splits.get("test")
    if split is None:
        print("No test split available; skipping evaluation.")
        return None

    if not os.path.exists(path):
        print(f"Model path not found: {path}")
        return None
    tokenizer_name = tokenizer_name or path
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    model = AutoModelForSequenceClassification.from_pretrained(path).to("cpu")
    all_preds, all_labels = [], []
    for i in range(len(split)):
        text = split[i]["text_content"]
        label = split[i]["label_int"]
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
        if "token_type_ids" in inputs: inputs.pop("token_type_ids")
        with torch.no_grad():
            outputs = model(**inputs)
        logits = outputs.logits
        pred = int(torch.argmax(logits, dim=-1))
        all_preds.append(pred); all_labels.append(label)
    return {"macro_f1": f1_score(all_labels, all_preds, average='macro'), "accuracy": accuracy_score(all_labels, all_preds)}

# Try evaluating Model A (baseline) if present
res = evaluate_saved_model(ARTIFACTS_DIR + "\\model_A_finetuned")
print("Model A evaluation:", res)

Model A evaluation: {'macro_f1': 0.8136526190426165, 'accuracy': 0.8131553860819828}


## Outputs & Observations
- If you have `models/model_A_finetuned/` etc., this notebook will evaluate them and produce reproducible metrics; otherwise, training cells demonstrate how to reproduce them in a controlled way.